In [1]:
import geopandas as gpd
import pandas as pd
import requests
import folium
import json

In [2]:
# GBFS discovery endpoint
# Check status and raw content first
url = "https://gbfs.bcycle.com/bcycle_lametro/station_information.json"
response = requests.get(url)
print(response.status_code)
print(response.text[:500])


200
{"ttl":60,"data":{"stations":[{"lon":-118.25854,"lat":34.04850,"rental_uris":{"ios":"https://www.bcycle.com/applink?system_id=bcycle_lametro&station_id=bcycle_lametro_3005&platform=iOS","android":"https://www.bcycle.com/applink?system_id=bcycle_lametro&station_id=bcycle_lametro_3005&platform=android"},"_bcycle_station_type":"Kiosk and Station","region_id":"bcycle_lametro_region_1","address":"700 Flower St","name":"7th & Flower","station_id":"bcycle_lametro_3005"},{"lon":-118.25667,"lat":34.04554


In [3]:
stations = response.json()
print(json.dumps(stations['data']['stations'][0], indent=2))

{
  "lon": -118.25854,
  "lat": 34.0485,
  "rental_uris": {
    "ios": "https://www.bcycle.com/applink?system_id=bcycle_lametro&station_id=bcycle_lametro_3005&platform=iOS",
    "android": "https://www.bcycle.com/applink?system_id=bcycle_lametro&station_id=bcycle_lametro_3005&platform=android"
  },
  "_bcycle_station_type": "Kiosk and Station",
  "region_id": "bcycle_lametro_region_1",
  "address": "700 Flower St",
  "name": "7th & Flower",
  "station_id": "bcycle_lametro_3005"
}


In [4]:
url = "https://gitlab.com/LACMTA/gtfs_rail/-/raw/master/gtfs_rail.zip"
response = requests.get(url)
print(response.status_code)
print(len(response.content), "bytes")

200
1125532 bytes


In [6]:
import os
import zipfile

# Create path if it doesn't exist
os.makedirs("data/raw", exist_ok=True)

# Save the zip file
with open("data/raw/gtfs_rail.zip", "wb") as f:
    f.write(response.content)
# Extract the zip
#with zipfile.ZipFile("data/raw/gtfs_rail.zip", "r") as zip_ref:
# See what files we got
 #   print(file)


# Download bus GTFS
bus_url = "https://gitlab.com/LACMTA/gtfs_bus/-/raw/master/gtfs_bus.zip"
response = requests.get(bus_url)

with open("data/raw/gtfs_bus.zip", "wb") as f:
    f.write(response.content)

with zipfile.ZipFile("data/raw/gtfs_bus.zip", "r") as zip_ref:
    zip_ref.extractall("data/raw/gtfs_bus")


In [7]:
# --- BIKE SHARE ---
stations_url = "https://gbfs.bcycle.com/bcycle_lametro/station_information.json"
status_url = "https://gbfs.bcycle.com/bcycle_lametro/station_status.json"

df_stations = pd.DataFrame(requests.get(stations_url).json()['data']['stations'])
df_status = pd.DataFrame(requests.get(status_url).json()['data']['stations'])

# --- GTFS RAIL ---
df_stops = pd.read_csv("data/raw/gtfs_rail/stops.txt")
df_routes = pd.read_csv("data/raw/gtfs_rail/routes.txt")
df_shapes = pd.read_csv("data/raw/gtfs_rail/shapes.txt")

#-- QUICK SUMMARY ---
print("=== BIKE SHARE ===")
print(f"Stations: {len(df_stations)} | Columns {list(df_stations.columns)}")
print (f"Status fields : {list (df_status.columns)}")
print("\n=== GTFS RAIL ===")
print(f"Stops: {len(df_stops)} | Columns: {list(df_stops.columns)}")
print(f"Routes: {len(df_routes)} | Columns: {list(df_routes.columns)}")
print(f"Shapes points: {len(df_shapes)}")
df_bus_stops = pd.read_csv("data/raw/gtfs_bus/stops.txt")
print(f"Bus stops: {len(df_bus_stops)}")


=== BIKE SHARE ===
Stations: 225 | Columns ['lon', 'lat', 'rental_uris', '_bcycle_station_type', 'region_id', 'address', 'name', 'station_id']
Status fields : ['is_returning', 'is_renting', 'is_installed', 'num_docks_available', 'num_bikes_available', 'last_reported', 'num_bikes_available_types', 'station_id']

=== GTFS RAIL ===
Stops: 448 | Columns: ['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'stop_url', 'location_type', 'parent_station', 'tpis_name']
Routes: 6 | Columns: ['route_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_color', 'route_text_color', 'route_url']
Shapes points: 16783
Bus stops: 11881


In [8]:
# Merge bike share stations wiht live status
df_bikes = df_stations.merge(df_status, on = 'station_id')
print(f"Merged Shaped {df_bikes.shape}")
print(df_bikes[['name', 'lat', 'lon', 'num_bikes_available', 'num_docks_available']].head(10))


Merged Shaped (225, 15)
                        name       lat        lon  num_bikes_available  \
0               7th & Flower  34.04850 -118.25854                   10   
1                Olive & 8th  34.04554 -118.25667                   19   
2                5th & Grand  34.05048 -118.25459                    9   
3             Figueroa & 9th  34.04661 -118.26273                    8   
4               11th & Maple  34.03705 -118.25487                    7   
5            Figueroa & 12th  34.04157 -118.26735                    7   
6  Union Station West Portal  34.05702 -118.23708                   18   
7       Los Angeles & Temple  34.05290 -118.24156                    7   
8            Grand & Olympic  34.04373 -118.26014                   11   
9                12th & Hill  34.03861 -118.26086                    4   

   num_docks_available  
0                   18  
1                   12  
2                   14  
3                    7  
4                    8  
5          

In [9]:
#check that the data is actually correct
# Check the timestamp of the status feed
status_raw = requests.get(status_url).json()
print(f"Last updated: {status_raw['last_updated']}")

import datetime
print(datetime.datetime.fromtimestamp(status_raw['last_updated']))


Last updated: 1772856721
2026-03-06 20:12:01


In [11]:
# Build shape_id → route_color lookup through trips
shape_to_route = df_trips[['route_id', 'shape_id']].drop_duplicates()
shape_to_route = shape_to_route.merge(
    df_routes[['route_id', 'route_long_name', 'route_color']], 
    on='route_id'
)
print(shape_to_route.head())

NameError: name 'df_trips' is not defined

In [10]:
m = folium.Map(
    location=[34.0522, -118.2437],
    zoom_start=11,
    tiles="CartoDB positron"
)

# --- LAYER 1: BUS STOPS ---
bus_layer = folium.FeatureGroup(name="Bus Stops", show=False)
for _, row in df_bus_stops.iterrows():
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=0.5,
        color='#AAAAAA',
        fill=True,
        fill_opacity=0.4,
    ).add_to(bus_layer)
bus_layer.add_to(m)

# --- LAYER 2: RAIL LINES in Metro colors ---
rail_layer = folium.FeatureGroup(name="Rail Lines", show=True)
for _, shape_row in shape_to_route.iterrows():
    shape_id = shape_row['shape_id']
    color = f"#{shape_row['route_color']}"
    line_name = shape_row['route_long_name']
    
    points = df_shapes[df_shapes['shape_id'] == shape_id]\
        .sort_values('shape_pt_sequence')[['shape_pt_lat', 'shape_pt_lon']]\
        .values.tolist()
    
    if points:
        folium.PolyLine(
            points,
            color=color,
            weight=4,
            opacity=0.9,
            tooltip=line_name
        ).add_to(rail_layer)
rail_layer.add_to(m)

# --- LAYER 3: RAIL STOPS ---
rail_stops_layer = folium.FeatureGroup(name="Rail Stops", show=True)
for _, row in df_stops_only.iterrows():
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=4,
        color='#FFFFFF',
        weight=2,
        fill=True,
        fill_color='#333333',
        fill_opacity=1,
        popup=row['stop_name']
    ).add_to(rail_stops_layer)
rail_stops_layer.add_to(m)

# --- LAYER 4: BIKE STATIONS ---
bike_layer = folium.FeatureGroup(name="Bike Share", show=True)
for _, row in df_bikes.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2,
        color='#E63946',
        fill=True,
        fill_opacity=0.5,
        popup=folium.Popup(
            f"<b>{row['name']}</b><br>"
            f"Bikes available: {row['num_bikes_available']}<br>"
            f"Docks available: {row['num_docks_available']}",
            max_width=200
        )
    ).add_to(bike_layer)
bike_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("data/processed/map_v3.html")
print("Map v3 saved")

NameError: name 'shape_to_route' is not defined

In [ ]:
print(f"Bus stops loaded: {len(df_bus_stops)}")
print(f"Rail routes: {len(df_routes)}")
print(df_routes[['route_short_name', 'route_long_name', 'route_color']])


#### Relate database

In [ ]:
df_trips = pd.read_csv("data/raw/gtfs_rail/trips.txt")
print(df_trips[['route_id', 'shape_id']].drop_duplicates().head(100))
#explain here,
# Build shape_id → route_color lookup through trips
shape_to_route = df_trips[['route_id', 'shape_id']].drop_duplicates()
shape_to_route = shape_to_route.merge(
    df_routes[['route_id', 'route_long_name', 'route_color']], 
    on='route_id'
)
print(shape_to_route.head())

### Census Data information


In [ ]:
# ============================================================
# BLOCK 6 — Census Demographic Data Pull
# American Community Survey (ACS) 5-year estimates, 2022
# Geography: All census tracts in LA County
# State FIPS: 06 (California), County FIPS: 037 (LA County)

load_dotenv()
CENSUS_KEY = os.getenv("CENSUS_API_KEY")

# ?get = what variables you want
# &for = what geography
# &in  = parent geography (state + county)
# &key = your API ke

census_url = (
    f"https://api.census.gov/data/2022/acs/acs5"
    f"?get=NAME,"
    f"B19013_001E,"  # median household income
    f"B19301_001E,"  # per capita income
    f"B19083_001E,"  # gini index (0=equal, 1=unequal)
    f"B17001_002E,"  # population below poverty line
    f"B01003_001E,"  # total population
    f"B25010_001E,"  # average household size
    f"B08201_002E,"  # households with no vehicle
    f"B08201_001E,"  # total households (denominator)
    f"B08301_001E,"  # total workers 16+ (denominator for mode share)
    f"B08301_010E,"  # workers commuting by transit
    f"B08301_016E,"  # workers commuting by bike
    f"B08301_019E,"  # workers commuting by foot
    f"B08301_021E,"  # workers working from home
    f"B08303_001E"   # mean travel time to work (minutes)
    f"&for=tract:*"              # every tract...
    f"&in=state:06%20county:037" # ...within CA, LA County
    f"&key={CENSUS_KEY}"
)

# fetch data
response = requests.get(census_url)
 # print(type(response))
#print should return 200, http means ok, <class 'requests.models.Response'>

census_data = response.json()
df_census = pd.DataFrame(
    census_data[1:],       # all rows except the header
    columns=census_data[0]
)

df_census = df_census.rename(columns={
    'B19013_001E': 'median_income',
    'B19301_001E': 'per_capita_income',
    'B19083_001E': 'gini_index',
    'B17001_002E': 'pop_below_poverty',
    'B01003_001E': 'total_population',
    'B25010_001E': 'avg_household_size',
    'B08201_002E': 'no_vehicle_households',
    'B08201_001E': 'total_households',
    'B08301_001E': 'total_workers',
    'B08301_010E': 'transit_commuters',
    'B08301_016E': 'bike_commuters',
    'B08301_019E': 'walk_commuters',
    'B08301_021E': 'wfh_commuters',
    'B08303_001E': 'mean_travel_time'
})

# --- CONVERT TO NUMERIC ---
# Census API returns everything as strings
# errors='coerce' turns invalid values (like -666666) into NaN

cols = [
    'median_income', 'per_capita_income', 'gini_index',
    'pop_below_poverty', 'total_population', 'avg_household_size',
    'no_vehicle_households', 'total_households', 'total_workers',
    'transit_commuters', 'bike_commuters', 'walk_commuters',
    'wfh_commuters', 'mean_travel_time'
]


df_census[cols] = df_census[cols].apply(pd.to_numeric, errors='coerce')

# --- DERIVED METRICS ---
# Raw counts are hard to compare across tracts of different sizes
# Percentages normalize the data — much more meaningful spatially
# Define all percentage calculations in one place
pct_calculations = {
    'pct_no_vehicle': ('no_vehicle_households', 'total_households'),
    'pct_transit':    ('transit_commuters',      'total_workers'),
    'pct_bike':       ('bike_commuters',          'total_workers'),
    'pct_walk':       ('walk_commuters',          'total_workers'),
    'pct_wfh':        ('wfh_commuters',           'total_workers'),
    'pct_poverty':    ('pop_below_poverty',        'total_population'),
}


for new_col, (numer, denom) in pct_calculations.items():
    df_census[new_col] = (
        df_census[numer] / df_census[denom] * 100
    ).round(1)

print(f"Census tracts: {len(df_census)}")
print(f"Columns: {list(df_census.columns)}")
print(df_census[[
    'NAME', 'median_income', 'gini_index',
    'pct_no_vehicle', 'pct_transit', 'pct_bike'
]].head(10))


In [ ]:
import geopandas as gpd

# --- DOWNLOAD CENSUS TRACT GEOMETRIES ---
# TIGER shapefiles from Census Bureau
# 2022 tract boundaries for California (state FIPS 06)
tiger_url = "https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_06_tract.zip"
response = requests.get(tiger_url)

with open("data/raw/ca_tracts.zip", "wb") as f:
    f.write(response.content)

with zipfile.ZipFile("data/raw/ca_tracts.zip", "r") as zip_ref:
    zip_ref.extractall("data/raw/ca_tracts")

print("Geometries downloaded")

# --- LOAD INTO GEODATAFRAME ---
gdf_tracts = gpd.read_file("data/raw/ca_tracts/tl_2022_06_tract.shp")

# Filter to LA County only (COUNTYFP = 037)
gdf_la = gdf_tracts[gdf_tracts['COUNTYFP'] == '037'].copy()
print(f"LA County tracts: {len(gdf_la)}")
print(f"Columns: {list(gdf_la.columns)}")
print(f"Coordinate system: {gdf_la.crs}")